# Measure the summer sale's lift

*Data Analysis in Python · Topic 6.1*

The store's summer sale ran from June 20 to July 7, 2025. Compare the sale's days with the same number of days just before it: did the sale bring more visitors, more orders, or both?

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

1. Run the parameters cell and the cell that builds the `sessions` table (lesson 3.'s steps).
2. In the first cell marked ✏️, create:
   - the `daily` table: one row per date of the `sessions` table, with the dates as the index, and the columns `sessions`, `transactions` and `revenue`
   - the `july` table: the rows of the `daily` table in July 2025
3. In the second cell marked ✏️, create:
   - the `sale` table: the rows of the `daily` table from `SALE_START` to `SALE_END`, both included
   - `sale_days`: the number of days in the sale, from `SALE_START` and `SALE_END`
   - the `before` table: the `sale_days` days just before `SALE_START`, built with `pd.Timedelta`
4. In the last cell marked ✏️, create:
   - `session_lift`: the `sale` table's sessions per day over the `before` table's, minus 1
   - `revenue_lift`: the same for revenue
   - `sale_rate` and `before_rate`: each table's conversion rate, its total transactions over its total sessions
   - `sale_weekend_days` and `before_weekend_days`: the number of Saturdays and Sundays in each table
5. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output.

**Expected output**: The `daily` table has 61 rows, June and July 2025, with the dates as its index, and the `july` table has 31. The `sale` table has 18 rows, June 20 to July 7, and `sale_days` is `18`. The `before` table has 18 rows, June 2 to 19. `session_lift` is about `0.4332` and `revenue_lift` about `0.6746`. `sale_rate` is about `0.0355` and `before_rate` about `0.0225`. The sale has 6 weekend days, and the `before` table has 4.

## Parameters

In [ ]:
import pandas as pd

DATA_URL = "https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/"
MONTHS = ["2025-06", "2025-07"]
START, END = "2025-06-01", "2025-07-31"
SALE_START, SALE_END = pd.Timestamp("2025-06-20"), pd.Timestamp("2025-07-07")

In [ ]:
# The clean sessions table, with lesson 3's steps. Run it, and leave it as it is.
DAY_FIRST_MONTHS = ["2025-03"]
SOURCE_ALIASES = {
    "facebook.com": "facebook", "m.facebook.com": "facebook", "l.facebook.com": "facebook",
    "lm.facebook.com": "facebook", "fb": "facebook",
    "instagram.com": "instagram", "l.instagram.com": "instagram", "ig": "instagram",
}


def read_export(month):
    """Read one month's export with dates as dates, revenue as numbers and lost campaigns as missing."""
    export = pd.read_csv(DATA_URL + f"sessions_export_{month.replace('-', '_')}.csv",
                         na_values={"session_campaign": ["(not set)", "-"]})
    date_format = "%d/%m/%Y" if month in DAY_FIRST_MONTHS else "%Y-%m-%d"
    export["date"] = pd.to_datetime(export["date"], format=date_format)
    if not pd.api.types.is_numeric_dtype(export["revenue"]):
        export["revenue"] = export["revenue"].str.replace("$", "").str.replace(",", "").astype(float)
    return export


raw = pd.concat([read_export(month) for month in MONTHS], ignore_index=True)
raw["session_campaign"] = raw["session_campaign"].fillna("(not set)")
deduped = raw.drop_duplicates()
sessions = deduped[deduped["date"].between(START, END) & (deduped["session_medium"] != "test")].reset_index(drop=True)
sessions["source"] = sessions["session_source"].str.lower().str.strip().replace(SOURCE_ALIASES)
sessions.shape

## One row per day

In [ ]:
daily = sessions.groupby("date")[["sessions", "transactions", "revenue"]].sum()
july = daily.loc["2025-07"]
daily.shape, july.shape

## The sale and the days before it

In [ ]:
sale = daily.loc[SALE_START:SALE_END]
sale_days = (SALE_END - SALE_START).days + 1
before = daily.loc[SALE_START - pd.Timedelta(days=sale_days):SALE_START - pd.Timedelta(days=1)]
sale.index[0], sale.index[-1], before.index[0], before.index[-1]

## The lift

In [ ]:
session_lift = sale["sessions"].mean() / before["sessions"].mean() - 1
revenue_lift = sale["revenue"].mean() / before["revenue"].mean() - 1
sale_rate = sale["transactions"].sum() / sale["sessions"].sum()
before_rate = before["transactions"].sum() / before["sessions"].sum()
sale_weekend_days = (sale.index.day_of_week >= 5).sum()
before_weekend_days = (before.index.day_of_week >= 5).sum()
session_lift, revenue_lift, sale_rate, before_rate, sale_weekend_days, before_weekend_days

In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables and the cells that have run. It doesn't change anything.
def check_your_work():
    import sys
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    else:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
        if checked_before:
            results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
        elif repeated:
            first_line = repeated[0].strip().splitlines()[0]
            results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
        elif error is not None:
            results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
        else:
            results.append((True, "Every cell in this session ran once, with no errors."))

    def number(value):
        return pd.api.types.is_number(value) and not isinstance(value, (pd.DataFrame, pd.Series))

    daily = found.get("daily")
    july = found.get("july")
    if not isinstance(daily, pd.DataFrame):
        results.append((False, "daily isn't a DataFrame yet. Total the sessions table by date: sessions.groupby(\"date\")[[\"sessions\", \"transactions\", \"revenue\"]].sum()."))
    elif not isinstance(daily.index, pd.DatetimeIndex):
        results.append((False, f"The daily table's index is a {type(daily.index).__name__}, not dates. Group by date without as_index=False, so the dates become the index."))
    elif len(daily) != 61 or not {"sessions", "transactions", "revenue"} <= set(daily.columns):
        results.append((False, f"The daily table should have 61 rows, one per day of June and July 2025, and the columns sessions, transactions and revenue. It has {len(daily):,} rows and {list(daily.columns)}."))
    elif daily["sessions"].sum() != 209453 or abs(daily["revenue"].sum() - 405061.22) > 0.01:
        results.append((False, f"The daily table's sessions and revenue add up to {daily['sessions'].sum():,} and ${daily['revenue'].sum():,.2f}, but the sessions table's are 209,453 and $405,061.22. Total them with sum()."))
    elif not isinstance(july, pd.DataFrame) or len(july) != 31 or july.index.min() != pd.Timestamp("2025-07-01"):
        results.append((False, "july should be the daily table's 31 rows of July 2025: daily.loc[\"2025-07\"]."))
    else:
        results.append((True, "The daily table has 61 days, June and July 2025, with the dates as its index."))

    sale = found.get("sale")
    sale_days = found.get("sale_days")
    before = found.get("before")
    sale_start, sale_end = pd.Timestamp("2025-06-20"), pd.Timestamp("2025-07-07")
    if not isinstance(sale, pd.DataFrame) or not isinstance(sale.index, pd.DatetimeIndex) or len(sale) == 0:
        results.append((False, "sale should be the daily table's rows from SALE_START to SALE_END: daily.loc[SALE_START:SALE_END]."))
    elif len(sale) == 17 and sale.index.max() == pd.Timestamp("2025-07-06"):
        results.append((False, "The sale table has 17 rows and ends on July 6. A slice by dates includes its end, so daily.loc[SALE_START:SALE_END] has July 7 too."))
    elif len(sale) != 18 or sale.index.min() != sale_start or sale.index.max() != sale_end:
        results.append((False, f"The sale table should have 18 rows, June 20 to July 7. It has {len(sale):,}, from {sale.index.min():%B %d} to {sale.index.max():%B %d}."))
    elif not number(sale_days):
        results.append((False, f"sale_days should be the number of days in the sale, 18. It is {sale_days!r}."))
    elif sale_days == 17:
        results.append((False, "sale_days is 17: SALE_END - SALE_START is the time between the first and last day, so it leaves one day out. Add 1, or use len(sale)."))
    elif sale_days != 18:
        results.append((False, f"sale_days should be 18, June 20 to July 7. It is {sale_days!r}."))
    elif not isinstance(before, pd.DataFrame) or not isinstance(before.index, pd.DatetimeIndex) or len(before) == 0:
        results.append((False, "before should be the daily table's sale_days days just before SALE_START, sliced with pd.Timedelta."))
    elif before.index.max() == sale_start:
        results.append((False, "The before table ends on June 20, the sale's first day, so the two windows share a day. End it the day before: SALE_START - pd.Timedelta(days=1)."))
    elif len(before) != 18:
        results.append((False, f"The before table has {len(before)} days, but the sale has 18. Start it SALE_START - pd.Timedelta(days=sale_days), so per-day numbers compare like with like."))
    elif before.index.min() != pd.Timestamp("2025-06-02") or before.index.max() != pd.Timestamp("2025-06-19"):
        results.append((False, f"The before table should be June 2 to 19. It's {before.index.min():%B %d} to {before.index.max():%B %d}."))
    else:
        results.append((True, "The sale has 18 days, June 20 to July 7, and the before table the 18 days just before it, June 2 to 19."))

    lifts = {name: found.get(name) for name in ["session_lift", "revenue_lift", "sale_rate", "before_rate", "sale_weekend_days", "before_weekend_days"]}
    missing = [name for name, value in lifts.items() if not number(value)]
    if missing:
        results.append((False, f"{', '.join(missing)} should be numbers. Check each one: {', '.join(f'{name} is {lifts[name]!r}' for name in missing[:2])}."))
    elif abs(lifts["session_lift"] - 1.433236) < 0.0001:
        results.append((False, "session_lift is 1.43: the sale's sessions per day over the before table's, without the minus 1. A lift is the ratio minus 1: 0.43 is 43% more."))
    elif abs(lifts["session_lift"] - 0.433236) > 0.0001 or abs(lifts["revenue_lift"] - 0.674573) > 0.0001:
        results.append((False, f"session_lift and revenue_lift should be the sale's per-day mean over the before table's, minus 1: about 0.4332 and 0.6746. They are {lifts['session_lift']:.4f} and {lifts['revenue_lift']:.4f}."))
    elif abs(lifts["sale_rate"] - 0.035561) < 0.000005 or abs(lifts["before_rate"] - 0.022588) < 0.000005:
        results.append((False, "sale_rate or before_rate is the mean of the days' rates. Divide each table's total transactions by its total sessions instead (topic 4.4)."))
    elif abs(lifts["sale_rate"] - 0.035484) > 0.000005 or abs(lifts["before_rate"] - 0.022548) > 0.000005:
        results.append((False, f"sale_rate and before_rate should be each table's transactions total over its sessions total: about 0.0355 and 0.0225. They are {lifts['sale_rate']:.4f} and {lifts['before_rate']:.4f}."))
    elif (lifts["sale_weekend_days"], lifts["before_weekend_days"]) == (3, 2):
        results.append((False, "The weekend counts are 3 and 2: those are the Sundays. day_of_week is 5 for Saturday and 6 for Sunday, so count day_of_week >= 5."))
    elif (lifts["sale_weekend_days"], lifts["before_weekend_days"]) != (6, 4):
        results.append((False, f"The sale has 6 Saturdays and Sundays, and the before table 4. Your counts are {lifts['sale_weekend_days']!r} and {lifts['before_weekend_days']!r}: (table.index.day_of_week >= 5).sum()."))
    else:
        results.append((True, f"During the sale, the store had {lifts['session_lift']:.0%} more sessions and {lifts['revenue_lift']:.0%} more revenue per day, and converted {lifts['sale_rate']:.2%} of sessions instead of {lifts['before_rate']:.2%}."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run.")


check_your_work()

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
daily = sessions.groupby("date")[["sessions", "transactions", "revenue"]].sum()
july = daily.loc["2025-07"]
daily.shape, july.shape
```

```python
sale = daily.loc[SALE_START:SALE_END]
sale_days = (SALE_END - SALE_START).days + 1
before = daily.loc[SALE_START - pd.Timedelta(days=sale_days):SALE_START - pd.Timedelta(days=1)]
sale.index[0], sale.index[-1], before.index[0], before.index[-1]
```

```python
session_lift = sale["sessions"].mean() / before["sessions"].mean() - 1
revenue_lift = sale["revenue"].mean() / before["revenue"].mean() - 1
sale_rate = sale["transactions"].sum() / sale["sessions"].sum()
before_rate = before["transactions"].sum() / before["sessions"].sum()
sale_weekend_days = (sale.index.day_of_week >= 5).sum()
before_weekend_days = (before.index.day_of_week >= 5).sum()
session_lift, revenue_lift, sale_rate, before_rate, sale_weekend_days, before_weekend_days
```

</details>